# Phase 1 · Foundations

**The question for this phase:** can I build one clean table with one row for each of Nigeria's
36 states and the FCT, where every number traces back to a dated source?

This notebook walks through what the pipeline does, step by step. It doesn't redo the work:
it calls the same code in `pipeline/` and reads what that code wrote to `data/processed/`.
So what you see here is exactly what the app and README will use.

Every number below is printed by code. None are typed in.

In [1]:
import sys, json, sqlite3
from pathlib import Path
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path[:0] = [str(ROOT / "pipeline"), str(ROOT / "src")]

import common, clean, join_states, validate
from chart_style import style, label_extremes, BLUE, ORANGE, NAVY, MUTED

# One renderer only: VS Code and JupyterLab draw this natively, no internet needed.
pio.renderers.default = "plotly_mimetype"
pd.set_option("display.max_rows", 60, "display.width", 160)

P = ROOT / "data" / "processed"
panel = pd.read_csv(P / "state_panel.csv")
dictionary = pd.read_csv(P / "state_panel_dictionary.csv")
meta = json.load(open(P / "meta.json", encoding="utf-8"))
print(f"Pipeline last built: {meta['built']} · state_panel rows: {len(panel)}")

Pipeline last built: 2026-11-24 · state_panel rows: 37


## 1. The sources

Each source is recorded in `data/sources.yaml` with who published it, when I accessed it and the
fields used. "To verify" means I haven't yet confirmed that detail on the publisher's own page.

In [2]:
src = common.load_sources()
pd.DataFrame([{"id": k, "name": v["name"], "data as of": v["data_as_of"],
               "accessed": v["date_accessed"], "licence": v["licence"]} for k, v in src.items()])

,id,name,data as of,accessed,licence
0,grid3_hf_v2,GRID3 NGA - Health Facilities v2.0,2024-11-11,2026-11-24,to verify (not confirmed on the dataset page)
1,ehealthafrica_hf,Nigeria: Health facilities,2018-12-18 to 2020-07-04,2026-11-24,to verify
2,grid3_pop_v3,GRID3 / WorldPop NGA population v3.0 (admin-le...,2025 estimates,2026-11-24,to verify
3,ndhs2024_api,Nigeria Demographic and Health Survey 2024 (NG...,fieldwork 2023-12-01 to 2024-05-01; published ...,set by pipeline/fetch_dhs.py,to verify (DHS Program terms of use)
4,ndhs2024_fr395,"Nigeria Demographic and Health Survey 2024, Fi...",fieldwork 2023-12-01 to 2024-05-01; published ...,2026-11-24,to verify
5,wdi,World Development Indicators (Nigeria),latest year per indicator,set by pipeline/fetch_wdi.py,CC BY 4.0


## 2. Trap 1: one state, many spellings

The same state is written differently in each source. Every name goes through one lookup table
(`data/lookups/states.csv`). If a source uses a spelling the lookup doesn't know, the pipeline stops
instead of guessing.

In [3]:
con = sqlite3.connect(f"file:{common.RAW_FILES['grid3_hf_v2'].as_posix()}?mode=ro", uri=True)
grid3_names = pd.read_sql('select distinct state from "NGA_health_facilities_v2_0"', con).state
con.close()
eha_names = pd.read_csv(common.RAW_FILES["ehealthafrica_hf"], usecols=["state_name"]).state_name.unique()
pop_names = pd.read_csv(P / "population_state.csv").state_as_published
dhs_names = pd.read_csv(P / "dhs_state_long.csv")  # already official; show the raw labels instead
raw_dhs = json.load(open(common.latest_raw("dhs_NG2024DHS_state"), encoding="utf-8"))["payload"]["Data"]
dhs_labels = pd.Series(sorted({d["CharacteristicLabel"] for d in raw_dhs if d["LevelRank"] == 2}))

rows = []
for source, names in [("GRID3", grid3_names), ("eHealth Africa", pd.Series(eha_names)),
                      ("Population", pop_names), ("DHS API", dhs_labels)]:
    official = common.to_official_state(pd.Series(names).reset_index(drop=True))
    for raw, off in zip(names, official):
        if str(raw).strip() != off:
            rows.append({"source": source, "as written": raw, "official name": off})
pd.DataFrame(rows)

,source,as written,official name
0,GRID3,Fct,Federal Capital Territory
1,eHealth Africa,Fct,Federal Capital Territory
2,Population,Cross-River,Cross River
3,Population,Federal Capital,Federal Capital Territory
4,Population,plateau,Plateau
5,DHS API,..Abia,Abia
6,DHS API,..Adamawa,Adamawa
7,DHS API,..Akwa Ibom,Akwa Ibom
8,DHS API,..Anambra,Anambra
9,DHS API,..Bauchi,Bauchi


## 3. Health facilities (GRID3 v2.0)

GRID3 calls its facility list **non-exhaustive and non-validated**: some facilities are missing,
some have closed, some appear twice. So these counts are a floor, not a full census.

### Trap 2: what "level" means
Each source labels levels its own way. All of them are mapped to primary, secondary or tertiary in one table:

In [4]:
pd.read_csv(common.LOOKUPS / "facility_levels.csv")

,source_id,field,raw_value,level,note
0,grid3_hf_v2,facility_level,Primary,primary,NaN
1,grid3_hf_v2,facility_level,Secondary,secondary,NaN
2,grid3_hf_v2,facility_level,Tertiary,tertiary,NaN
3,grid3_hf_v2,facility_level,Unknown,unknown,level not recorded in source
4,ehealthafrica_hf,type,Primary,primary,NaN
5,ehealthafrica_hf,type,Secondary,secondary,NaN
6,ehealthafrica_hf,type,Tertiary,tertiary,NaN
7,ehealthafrica_hf,type,Primary Health Clinic,primary,facility-type label used in the level field
8,ehealthafrica_hf,type,Primary Health Center,primary,facility-type label used in the level field
9,ehealthafrica_hf,type,Health Post Dispensary,primary,facility-type label used in the level field


In [5]:
g = pd.read_csv(P / "facilities_grid3.csv")
print("GRID3 facilities by level (after de-duplication):")
print(g.level.value_counts().to_string())
print("\nBy ownership:")
print(g.ownership.value_counts().to_string())

GRID3 facilities by level (after de-duplication):
level
primary      44905
secondary     5263
tertiary       506
unknown        334

By ownership:
ownership
public     34673
private    11710
unknown     4625


### Trap 3: duplicates
A row is dropped only if it has the same state, LGA and name (letters and digits only, so
"P.H.C" matches "PHC") **and** sits within the distance below of an earlier row.

In [6]:
dropped = pd.read_csv(P / "facilities_grid3_dropped_duplicates.csv")
s = meta["pipeline_steps"]["grid3"]
print(f"Distance rule: under {clean.DEDUP_METRES} m")
print(f"Rows in: {s['rows_in']:,} · dropped: {s['duplicates_dropped']} · kept: {s['rows_out']:,}")
dropped[["state", "lga", "facility_name", "level", "latitude", "longitude"]]

Distance rule: under 100 m
Rows in: 51,022 · dropped: 14 · kept: 51,008


,state,lga,facility_name,level,latitude,longitude
0,Ekiti,Ikole,Ikole Methodist Primary Health Care Center,primary,7.793610,5.515350
1,Enugu,Enugu South,Divine Providence Hospital,primary,6.417530,7.492780
2,Enugu,Enugu South,Ugwuagba Health Center,primary,6.359480,7.507840
3,Federal Capital Territory,Abaji,Abaji General Hospital,secondary,8.486525,6.944094
4,Imo,Ideato South,Dikenafai Maternity and Child Health Centre,primary,5.765686,7.155701
5,Imo,Obowo,Umuagu Health Center,primary,5.567920,7.399940
6,Imo,Orlu,St Mary'S Children and Community Hospital,secondary,5.765460,7.039350
7,Kwara,Oke-Ero,Erinmope Health Center,primary,8.075160,5.090940
8,Lagos,Alimosho,Ayis Hospital,primary,6.551020,3.277010
9,Lagos,Alimosho,Blue Sea Hospital,primary,6.558550,3.279920


## 4. Facilities per 100,000 people

**Per 100,000** means: if every state had 100,000 people, how many facilities would it have?
It lets a big state and a small state be compared fairly.

I'm showing **secondary and tertiary** facilities (general hospitals and above), because that's
where specialist care like dialysis could sit. Rates are rounded to one decimal place, because
the population figures are modelled estimates (Trap 4).

In [7]:
d = panel.sort_values("fac_sec_tert_per_100k")
low, high = d.iloc[0], d.iloc[-1]
ratio = (high.fac_sec_tert_grid3 / high.population_total) / (low.fac_sec_tert_grid3 / low.population_total)  # unrounded rates
colors = [ORANGE if s in d.state.head(5).tolist() else BLUE for s in d.state]
names = d.state.replace({"Federal Capital Territory": "FCT"})
fig = go.Figure(go.Bar(x=d.fac_sec_tert_per_100k, y=names, orientation="h", marker_color=colors,
                       text=d.fac_sec_tert_per_100k.map("{:.1f}".format), textposition="outside",
                       textfont=dict(size=11, color=NAVY), cliponaxis=False,
                       hovertemplate="%{y}: %{x:.1f} per 100,000<extra></extra>"))
style(fig,
      title=f"{high.state} has about {ratio:.0f}x as many hospitals per person as {low.state}",
      subtitle="Secondary and tertiary health facilities per 100,000 people (lowest five in orange)",
      source="GRID3 NGA Health Facilities v2.0; GRID3/WorldPop population v3.0",
      height=950, x_title="Facilities per 100,000 people", x_grid=True)
fig.update_yaxes(tickfont=dict(size=12))
fig.show()

## 5. Cross-check: GRID3 vs eHealth Africa

I compared facility counts state by state against a second dataset, eHealth Africa on HDX,
and flagged any state where they differ by more than 20%.

**An important catch:** most of GRID3's coordinates come *from* eHealth Africa, so the two lists
aren't independent. Agreement between them is weaker evidence than it looks.

In [8]:
share = (g.geocoordinates_source == "GRID3_EHEALTH").mean()
print(f"Share of GRID3 v2.0 coordinates sourced from eHealth Africa: {share:.0%}")
cc = pd.read_csv(common.EXPORTS / "facility_crosscheck_grid3_vs_ehealthafrica.csv")
print(f"States differing by >{join_states.CROSSCHECK_THRESHOLD_PCT}%, all levels: "
      f"{int(cc.crosscheck_flag.sum())} of {len(cc)}")
print(f"States differing by >{join_states.CROSSCHECK_THRESHOLD_PCT}%, secondary + tertiary only: "
      f"{int(cc.crosscheck_sec_tert_flag.sum())} of {len(cc)}")

Share of GRID3 v2.0 coordinates sourced from eHealth Africa: 81%
States differing by >20%, all levels: 4 of 37
States differing by >20%, secondary + tertiary only: 36 of 37


In [9]:
from plotly.subplots import make_subplots
c2 = cc.sort_values("crosscheck_sec_tert_diff_percent").assign(
    name=lambda t: t.state.replace({"Federal Capital Territory": "FCT"}))
fig = make_subplots(rows=1, cols=2, shared_yaxes=True, horizontal_spacing=0.06,
                    subplot_titles=("All facilities", "Secondary + tertiary only"))
for col, (field, flag) in enumerate([("crosscheck_diff_percent", "crosscheck_flag"),
                                     ("crosscheck_sec_tert_diff_percent", "crosscheck_sec_tert_flag")], start=1):
    fig.add_trace(go.Scatter(x=c2[field], y=c2.name, mode="markers",
                             marker=dict(size=9, color=[ORANGE if f else BLUE for f in c2[flag]]),
                             hovertemplate="%{y}: %{x:+.0f}%<extra></extra>"), row=1, col=col)
    fig.add_vrect(x0=-20, x1=20, fillcolor="#E6E8EC", opacity=0.6, line_width=0, layer="below", row=1, col=col)
fig.update_annotations(font=dict(size=13, color=NAVY), selector=dict(text="All facilities"))
fig.update_annotations(font=dict(size=13, color=NAVY), selector=dict(text="Secondary + tertiary only"))
style(fig,
      title=(f"The two lists agree on total facilities in {int((~cc.crosscheck_flag).sum())} of {len(cc)} states, "
             f"but on hospitals in only {int((~cc.crosscheck_sec_tert_flag).sum())}"),
      subtitle="% difference in count, GRID3 v2.0 vs eHealth Africa. Grey band = within ±20%; orange = outside it",
      source="GRID3 NGA Health Facilities v2.0; eHealth Africa via HDX", height=950, x_grid=True)
fig.update_xaxes(title_text="% difference", ticksuffix="%")
fig.update_yaxes(tickfont=dict(size=12))
fig.show()

**What this means:** the two sources classify levels differently, so the hospital count I use for the
supply gap depends on which source I trust. GRID3 is newer (2024 vs 2018–2020) and is the main
source; eHealth Africa is kept as the low end of the range. This feeds straight into the
"Under the hood" notes.

## 6. Population

Nigeria hasn't had a census since 2006, so these are **modelled estimates** (2025, scaled to UN
projections), not counts. This file gives totals only, with no age groups, so I can't yet compute
population aged 40+, which is the kidney-disease demand measure in the brief. It stays `unknown`.

In [10]:
pop = pd.read_csv(P / "population_state.csv").sort_values("population_total", ascending=False)
pp = meta["pipeline_steps"]["population"]
print(f"Sum of states: {pp['sum_of_states']:,.0f} · national total in file: {pp['national_total']:,.0f}")
top = pop.head(5)[["state", "population_total"]].assign(population_total=lambda t: t.population_total.round(-3).map("{:,.0f}".format))
print("\nFive largest states:"); print(top.to_string(index=False))
print("\nFlag, Ogun above Lagos:", meta["flags"]["population_ogun_above_lagos"])

Sum of states: 237,527,782 · national total in file: 237,527,782

Five largest states:
  state population_total
   Kano       18,771,000
   Ogun       12,934,000
  Lagos       12,583,000
 Kaduna       11,233,000
Katsina       10,523,000

Flag, Ogun above Lagos: True


## 7. NDHS 2024: who can pay, and what holds people back

From the DHS Program API (survey `NG2024DHS`, fieldwork December 2023 to May 2024):
- **Health insurance:** share of women and men aged 15–49 with any insurance
  (for women, the API only gives "no insurance", so "any" = 100 minus that)
- **Wealth:** share of the household population in the top two national wealth quintiles
- **Barriers:** women who say distance, or getting money, is a big problem in getting care

In [11]:
cols = ["state", "insurance_any_women_pct", "insurance_any_men_pct", "wealth_top2_pct",
        "barrier_distance_women_pct", "barrier_money_women_pct"]
panel[cols].sort_values("insurance_any_women_pct").reset_index(drop=True)

,state,insurance_any_women_pct,insurance_any_men_pct,wealth_top2_pct,barrier_distance_women_pct,barrier_money_women_pct
0,Zamfara,0.2,0.4,14.5,7.3,54.6
1,Taraba,0.9,2.2,15.4,20.0,60.0
2,Ebonyi,1.0,1.9,18.2,38.2,52.6
3,Sokoto,1.0,2.4,13.2,48.2,61.5
4,Kebbi,1.1,0.8,7.1,43.0,54.1
5,Adamawa,1.1,1.1,14.0,48.3,72.1
6,Rivers,1.2,6.1,78.7,21.2,59.3
7,Niger,1.2,1.1,21.6,30.5,54.5
8,Kwara,1.5,4.4,53.6,23.9,39.5
9,Plateau,1.5,3.1,21.9,60.0,76.0


### Blood pressure and blood sugar: read with care
These come from FR395 Tables 19.1.1–19.2.2, transcribed with the page number on every row.
They're **self-reported** ("ever told by a health worker"), not measured, and cover ages 15–49 only.

The chart shows why that matters: you can only be *told* you have high blood pressure if someone
*measured* it. States where few people have ever been checked will look healthier than they are.

In [12]:
r = panel[["bp_ever_measured_women_pct", "bp_ever_told_high_women_pct"]].corr().iloc[0, 1]
x, y = "bp_ever_measured_women_pct", "bp_ever_told_high_women_pct"
fig = px.scatter(panel, x=x, y=y, text=label_extremes(panel, x, y, n=3), hover_name="state")
fig.update_traces(marker=dict(color=BLUE, size=10, opacity=0.85), textposition="top center",
                  textfont=dict(size=11, color=NAVY), hovertemplate="%{hovertext}<br>measured %{x}%<br>told high %{y}%<extra></extra>")
style(fig,
      title=(f"Where more women get checked, more are told their blood pressure is high (r = {r:.2f})" if r >= 0.3
             else f"Being checked more doesn't clearly mean being told more (r = {r:.2f})"),
      subtitle="Women 15–49 by state: % ever had blood pressure measured vs % ever told it was high",
      source="NDHS 2024 (FR395), Table 19.1.1", height=620,
      x_title="% ever had blood pressure measured", y_title="% ever told it was high")
fig.update_xaxes(ticksuffix="%"); fig.update_yaxes(ticksuffix="%")
fig.show()

## 8. An early look at the tension (not the scorecard)

The brief asks me to test, not assume, that **places with the greatest need may have the least
ability to pay.** The scorecard in Phase 2 does this properly. Here's a first look at one piece:
do states with fewer hospitals per person also have fewer better-off households?

Only the states at the edges are labelled, so the names don't pile up. Hover over any dot to see its state.

In [13]:
r = panel[["fac_sec_tert_per_100k", "wealth_top2_pct"]].corr(method="spearman").iloc[0, 1]
x, y = "fac_sec_tert_per_100k", "wealth_top2_pct"
fig = px.scatter(panel, x=x, y=y, text=label_extremes(panel, x, y, n=2), hover_name="state")
fig.update_traces(marker=dict(color=BLUE, size=10, opacity=0.85), textposition="top center",
                  textfont=dict(size=11, color=NAVY),
                  hovertemplate="%{hovertext}<br>%{x:.1f} per 100,000<br>%{y:.0f}% in top two quintiles<extra></extra>")
style(fig,
      title=(f"States with fewer hospitals per person also tend to be poorer (rank correlation {r:.2f})" if r >= 0.3
             else f"States with fewer hospitals per person tend to be richer (rank correlation {r:.2f})" if r <= -0.3
             else f"No clear link between hospitals per person and wealth (rank correlation {r:.2f})"),
      subtitle="By state: secondary + tertiary facilities per 100,000 vs % of people in the top two wealth quintiles",
      source="GRID3 v2.0; GRID3/WorldPop v3.0; NDHS 2024 (DHS API)", height=620,
      x_title="Hospitals per 100,000 people", y_title="% in top two wealth quintiles")
fig.update_yaxes(ticksuffix="%")
fig.show()

## 9. Validation

The pipeline stops if any of these rules break. Running them again here:

In [14]:
print(validate.__doc__)
errors = validate.run_all_checks()
print("All checks passed." if not errors else "\n".join(errors))

Stop the run (exit code 1) if the data breaks any rule.

Rules:
 1. state_panel.csv has exactly 37 rows, one per state code in the lookup.
 2. Pillar input columns have no nulls (a real gap must be written "unknown").
 3. Every panel column is in the dictionary with a source_id and a data date,
    and every source_id is in data/sources.yaml.
 4. Every manual row has source_id, table and printed_page.
 5. Population > 0; percentage columns between 0 and 100.
 6. Sum of state populations within 1% of the national total.
 7. The de-duplication report and the GRID3 vs eHealth Africa comparison exist.
 8. Every register row has a source URL and source date   (active from phase 3).
 9. Payer-mix shares sum to 100%                          (active from phase 4).



All checks passed.


## 10. Gaps carried forward

These are written into `meta.json` by the pipeline and will become "Under the hood" notes.

In [15]:
for k, v in meta["flags"].items():
    print(f"- {k}: {v}")

- crosscheck_states_over_20pct: ['Plateau', 'Kwara', 'Jigawa', 'Kaduna']
- crosscheck_sec_tert_states_over_20pct: ['Adamawa', 'Zamfara', 'Katsina', 'Sokoto', 'Benue', 'Cross River', 'Enugu', 'Taraba', 'Nasarawa', 'Imo', 'Akwa Ibom', 'Ondo', 'Abia', 'Osun', 'Bayelsa', 'Rivers', 'Kano', 'Niger', 'Bauchi', 'Kogi', 'Edo', 'Kebbi', 'Yobe', 'Ebonyi', 'Anambra', 'Federal Capital Territory', 'Ekiti', 'Ogun', 'Gombe', 'Delta', 'Oyo', 'Lagos', 'Plateau', 'Kwara', 'Jigawa', 'Kaduna']
- population_40plus: unknown: source file has totals only
- population_ogun_above_lagos: True
- ndhs_bp_sugar: self-reported diagnosis, ages 15-49; partly measures access to testing
- ndhs_state_confidence_intervals: not available from API; FR395 Appendix B has state sampling errors only for a few indicators, not insurance or wealth
